# Phase 4 — CPCB AQI Calculation
### Notebook: `phase_4_aqi.ipynb`

**What this notebook does:**
1. Loads `outputs/air_pollution_master.csv` (city × month)
2. Applies official CPCB AQI breakpoint tables to compute a sub-index per pollutant
3. Takes the maximum sub-index as the final AQI for each city-month
4. Assigns the official CPCB 6-category label and the project 3-tier label
5. Saves `outputs/air_pollution_with_aqi.csv`
6. Aggregates AQI to state × year level
7. Merges AQI state-year columns into `combined_master.csv` and resaves it

---
**Official CPCB AQI formula (linear interpolation):**
```
Ip = [(IHi - ILo) / (BHi - BLo)] × (Cp - BLo) + ILo

where:
  Ip  = sub-index for pollutant p
  Cp  = measured concentration of pollutant p
  BHi = upper concentration breakpoint >= Cp
  BLo = lower concentration breakpoint <= Cp
  IHi = AQI value at BHi
  ILo = AQI value at BLo
```
Final AQI = maximum sub-index across all available pollutants.

---
**CRITICAL LIMITATION — must be stated in the project report:**  
CPCB AQI is designed for **24-hour averages** (PM2.5, PM10, NO2, SO2, NH3) and **8-hour averages** (CO, Ozone).  
Our data contains **monthly averages** computed in Phase 1.  
Monthly means are lower than daily peaks — AQI from monthly data will underestimate actual daily AQI.  
This is acceptable for **trend analysis and state comparisons** (relative rankings are valid),  
but should NOT be reported as equivalent to official daily CPCB AQI values.

---
**Pollutants used for AQI (7 of the official 8):**  
PM2.5, PM10, NO2, SO2, CO, NH3, Ozone  
Lead (Pb) is excluded — not measured in the CPCB station network dataset.

---
**Input:** `outputs/air_pollution_master.csv`  
**Input:** `outputs/combined_master.csv` (for updating with AQI state-year columns)  
**Output:** `outputs/air_pollution_with_aqi.csv`  
**Output:** `outputs/combined_master.csv` (updated, resaved)

---
## Cell 1 — Import Libraries

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

print("Libraries imported")

Libraries imported


---
## Cell 2 — Set Paths and Load Data

In [2]:
BASE_DIR   = Path('..')
OUTPUT_DIR = BASE_DIR / 'outputs'

POLLUTION_MASTER_PATH = OUTPUT_DIR / 'air_pollution_master.csv'
COMBINED_MASTER_PATH  = OUTPUT_DIR / 'combined_master.csv'

for fpath in [POLLUTION_MASTER_PATH, COMBINED_MASTER_PATH]:
    if not fpath.exists():
        raise FileNotFoundError(
            f"File not found: {fpath}\n"
            "Run Phase 1, 2, and 3 first."
        )

df_pollution = pd.read_csv(POLLUTION_MASTER_PATH, low_memory=False)
df_combined  = pd.read_csv(COMBINED_MASTER_PATH,  low_memory=False)

print(f"air_pollution_master.csv : {df_pollution.shape[0]:,} rows x {df_pollution.shape[1]} columns")
print(f"combined_master.csv      : {df_combined.shape[0]:,} rows x {df_combined.shape[1]} columns")

air_pollution_master.csv : 10,295 rows x 97 columns
combined_master.csv      : 18,228 rows x 43 columns


---
## Cell 3 — Define CPCB AQI Breakpoint Tables

**Source:** Table 3.1 from Ministry of Earth Sciences Standard Operating Procedure (SOP),  
citing CPCB National Air Quality Index (2014).  
These are the official government-mandated breakpoints — do not modify.

Each entry is a tuple: `(BLo, BHi, ILo, IHi)`  
- BLo / BHi = lower / upper concentration boundary for this band  
- ILo / IHi = lower / upper AQI value for this band  

The table uses **continuous boundaries** (e.g. 0–30, 30–60) rather than the integer  
boundaries in the official document (0–30, 31–60), because our data contains float  
monthly averages, not integer daily readings.

| AQI Category | Range | PM2.5 | PM10 | NO2 | SO2 | CO (mg/m³) | NH3 | Ozone |
|---|---|---|---|---|---|---|---|---|
| Good | 0–50 | 0–30 | 0–50 | 0–40 | 0–40 | 0–1.0 | 0–200 | 0–50 |
| Satisfactory | 51–100 | 31–60 | 51–100 | 41–80 | 41–80 | 1.1–2.0 | 201–400 | 51–100 |
| Moderate | 101–200 | 61–90 | 101–250 | 81–180 | 81–380 | 2.1–10 | 401–800 | 101–168 |
| Poor | 201–300 | 91–120 | 251–350 | 181–280 | 381–800 | 10.1–17 | 801–1200 | 169–208 |
| Very Poor | 301–400 | 121–250 | 351–430 | 281–400 | 801–1600 | 17.1–34 | 1201–1800 | 209–748 |
| Severe | 401–500 | 251+ | 431+ | 401+ | 1601+ | 34.1+ | 1801+ | 749+ |

In [3]:
# ── CPCB_BREAKPOINTS ──────────────────────────────────────────────────────────
# Format per entry: (BLo, BHi, ILo, IHi)
# Keys MUST match the exact column names in air_pollution_master.csv
# (i.e. the canonical names after Phase 1's DEDUPLICATE_MAP was applied)

INF = float('inf')   # Shorthand for 'above the highest defined breakpoint'

CPCB_BREAKPOINTS = {

    # ── PM2.5 (µg/m³, 24-hr standard) ────────────────────────────────────────
    'PM2.5 (ug/m3)': [
        (0,   30,   0,   50),
        (30,  60,   50,  100),
        (60,  90,   100, 200),
        (90,  120,  200, 300),
        (120, 250,  300, 400),
        (250, INF,  400, 500),
    ],

    # ── PM10 (µg/m³, 24-hr standard) ─────────────────────────────────────────
    'PM10 (ug/m3)': [
        (0,   50,   0,   50),
        (50,  100,  50,  100),
        (100, 250,  100, 200),
        (250, 350,  200, 300),
        (350, 430,  300, 400),
        (430, INF,  400, 500),
    ],

    # ── NO2 (µg/m³, 24-hr standard) ──────────────────────────────────────────
    'NO2 (ug/m3)': [
        (0,   40,   0,   50),
        (40,  80,   50,  100),
        (80,  180,  100, 200),
        (180, 280,  200, 300),
        (280, 400,  300, 400),
        (400, INF,  400, 500),
    ],

    # ── SO2 (µg/m³, 24-hr standard) ──────────────────────────────────────────
    'SO2 (ug/m3)': [
        (0,    40,   0,   50),
        (40,   80,   50,  100),
        (80,   380,  100, 200),
        (380,  800,  200, 300),
        (800,  1600, 300, 400),
        (1600, INF,  400, 500),
    ],

    # ── CO (mg/m³, 8-hr standard) ─────────────────────────────────────────────
    # NOTE: CO in our data is already in mg/m³ (confirmed from Phase 1
    # DEDUPLICATE_MAP — canonical name is 'CO (mg/m3)'). No conversion needed.
    'CO (mg/m3)': [
        (0,   1.0,  0,   50),
        (1.0, 2.0,  50,  100),
        (2.0, 10,   100, 200),
        (10,  17,   200, 300),
        (17,  34,   300, 400),
        (34,  INF,  400, 500),
    ],

    # ── NH3 (µg/m³, 24-hr standard) ──────────────────────────────────────────
    'NH3 (ug/m3)': [
        (0,    200,  0,   50),
        (200,  400,  50,  100),
        (400,  800,  100, 200),
        (800,  1200, 200, 300),
        (1200, 1800, 300, 400),
        (1800, INF,  400, 500),
    ],

    # ── Ozone (µg/m³, 8-hr standard) ─────────────────────────────────────────
    'Ozone (ug/m3)': [
        (0,   50,   0,   50),
        (50,  100,  50,  100),
        (100, 168,  100, 200),
        (168, 208,  200, 300),
        (208, 748,  300, 400),
        (748, INF,  400, 500),
    ],
}

# ── Verify all breakpoint keys exist as _mean columns in pollution master ──────
# WHY: If a key doesn't exist in the data, we'll silently skip that pollutant.
# Better to know now than to compute AQI with missing sub-indices.
print("Breakpoint pollutants vs available _mean columns:")
for pollutant in CPCB_BREAKPOINTS:
    mean_col = f'{pollutant}_mean'
    exists = mean_col in df_pollution.columns
    status = "OK" if exists else "MISSING in data"
    print(f"   [{status}]  {mean_col}")

print(f"\nTotal pollutants with breakpoints defined: {len(CPCB_BREAKPOINTS)}")

Breakpoint pollutants vs available _mean columns:
   [OK]  PM2.5 (ug/m3)_mean
   [OK]  PM10 (ug/m3)_mean
   [OK]  NO2 (ug/m3)_mean
   [OK]  SO2 (ug/m3)_mean
   [OK]  CO (mg/m3)_mean
   [OK]  NH3 (ug/m3)_mean
   [OK]  Ozone (ug/m3)_mean

Total pollutants with breakpoints defined: 7


---
## Cell 4 — Define AQI Computation Functions

Three functions:
1. `compute_sub_index(concentration, breakpoints)` — applies the CPCB linear interpolation formula to one pollutant value
2. `compute_row_aqi(row)` — takes one city-month row, computes all sub-indices, returns the max (= final AQI)
3. `aqi_to_category(aqi)` and `aqi_to_project_tier(aqi)` — convert numeric AQI to category labels

In [4]:
def compute_sub_index(concentration, breakpoints):
    """
    WHY THIS FUNCTION EXISTS:
    Applies the official CPCB linear interpolation formula to compute the
    AQI sub-index for a single pollutant at a given concentration.

    Formula: Ip = [(IHi - ILo) / (BHi - BLo)] × (Cp - BLo) + ILo

    Parameters
    ----------
    concentration : float — the measured monthly mean concentration
    breakpoints   : list of (BLo, BHi, ILo, IHi) tuples

    Returns
    -------
    float — the sub-index value (0 to 500), or np.nan if input is NaN
    """
    # Return NaN for missing data — cannot compute AQI without a value
    if pd.isna(concentration):
        return np.nan

    # Negative concentrations are physically impossible — treat as missing
    if concentration < 0:
        return np.nan

    # Find which breakpoint band this concentration falls in
    for (BLo, BHi, ILo, IHi) in breakpoints:
        if concentration <= BHi:
            # Guard against division by zero (should never happen with these tables)
            if BHi == BLo:
                return float(IHi)
            # Apply the linear interpolation formula
            sub_index = ((IHi - ILo) / (BHi - BLo)) * (concentration - BLo) + ILo
            # Clamp to valid range 0-500
            return float(np.clip(sub_index, 0, 500))

    # Concentration is above the last defined breakpoint — cap at 500
    return 500.0


def compute_row_aqi(row):
    """
    WHY THIS FUNCTION EXISTS:
    Takes one city-month row and computes the final AQI for it.

    Steps:
      1. For each pollutant with a breakpoint table, read its _mean value
      2. Compute the sub-index using compute_sub_index()
      3. Check that we have >= 3 sub-indices AND at least one of PM2.5 or PM10
         (CPCB rule: AQI requires minimum 3 pollutants, one must be particulate)
      4. Return the maximum sub-index as the final AQI

    Returns
    -------
    dict with keys: 'AQI', 'AQI_dominant_pollutant', 'AQI_n_pollutants_used'
    """
    sub_indices = {}   # pollutant_name → sub_index value

    for pollutant, breakpoints in CPCB_BREAKPOINTS.items():
        mean_col = f'{pollutant}_mean'
        if mean_col not in row.index:
            continue   # This pollutant column doesn't exist in the data
        concentration = row[mean_col]
        si = compute_sub_index(concentration, breakpoints)
        if not pd.isna(si):
            sub_indices[pollutant] = si

    # ── CPCB minimum data requirement ─────────────────────────────────────────
    # Need >= 3 pollutants, and at least one must be PM2.5 or PM10
    has_particulate = ('PM2.5 (ug/m3)' in sub_indices) or ('PM10 (ug/m3)' in sub_indices)
    enough_pollutants = len(sub_indices) >= 3

    if not has_particulate or not enough_pollutants:
        # Cannot compute a valid CPCB AQI for this row
        return {
            'AQI'                    : np.nan,
            'AQI_dominant_pollutant' : None,
            'AQI_n_pollutants_used'  : len(sub_indices),
        }

    # ── Final AQI = maximum sub-index ─────────────────────────────────────────
    dominant = max(sub_indices, key=sub_indices.get)
    final_aqi = sub_indices[dominant]

    return {
        'AQI'                    : round(final_aqi, 1),
        'AQI_dominant_pollutant' : dominant,
        'AQI_n_pollutants_used'  : len(sub_indices),
    }


def aqi_to_category(aqi):
    """
    Maps numeric AQI to the official CPCB 6-category label.
    Source: CPCB National Air Quality Index, 2014.
    """
    if pd.isna(aqi):  return 'Unknown'
    if aqi <= 50:     return 'Good'
    if aqi <= 100:    return 'Satisfactory'
    if aqi <= 200:    return 'Moderate'
    if aqi <= 300:    return 'Poor'
    if aqi <= 400:    return 'Very Poor'
    return 'Severe'


def aqi_to_project_tier(aqi):
    """
    Maps numeric AQI to the project's simplified 3-tier label.
    Good + Satisfactory → 'Good'  (AQI 0–100)
    Moderate + Poor     → 'Bad'   (AQI 101–300)
    Very Poor + Severe  → 'Worst' (AQI 301–500)
    """
    if pd.isna(aqi):  return 'Unknown'
    if aqi <= 100:    return 'Good'
    if aqi <= 300:    return 'Bad'
    return 'Worst'


# ── Quick unit test on compute_sub_index ─────────────────────────────────────
# Verifying against the confirmed example from CPCB documentation:
# PM2.5 = 45 µg/m³ → sub-index should be 75
test_val = compute_sub_index(45.0, CPCB_BREAKPOINTS['PM2.5 (ug/m3)'])
expected = 75.0
status   = "PASS" if abs(test_val - expected) < 0.5 else "FAIL"
print(f"Unit test — PM2.5=45 µg/m³ → sub-index={test_val:.1f} (expected ~75) [{status}]")

# Additional spot checks
print(f"PM2.5=31 → sub-index={compute_sub_index(31, CPCB_BREAKPOINTS['PM2.5 (ug/m3)']):.1f} (expected 51.7)")
print(f"PM2.5=60 → sub-index={compute_sub_index(60, CPCB_BREAKPOINTS['PM2.5 (ug/m3)']):.1f} (expected 100.0)")
print(f"PM2.5=0  → sub-index={compute_sub_index(0,  CPCB_BREAKPOINTS['PM2.5 (ug/m3)']):.1f} (expected 0.0)")
print(f"PM2.5=300→ sub-index={compute_sub_index(300,CPCB_BREAKPOINTS['PM2.5 (ug/m3)']):.1f} (expected 500.0)")
print()
print("All AQI functions defined")

Unit test — PM2.5=45 µg/m³ → sub-index=75.0 (expected ~75) [PASS]
PM2.5=31 → sub-index=51.7 (expected 51.7)
PM2.5=60 → sub-index=100.0 (expected 100.0)
PM2.5=0  → sub-index=0.0 (expected 0.0)
PM2.5=300→ sub-index=400.0 (expected 500.0)

All AQI functions defined


---
## Cell 5 — Compute AQI for Every City-Month Row

We apply `compute_row_aqi()` to every row in `air_pollution_master.csv`.  
This adds three new columns: `AQI`, `AQI_dominant_pollutant`, `AQI_n_pollutants_used`.  
Then we add `AQI_category` and `AQI_project_tier` from the two label functions.

**Expected runtime:** a few seconds for ~30,000–40,000 rows.

In [5]:
print("Computing AQI for every city-month row...")

# ── Apply compute_row_aqi to every row ────────────────────────────────────────
# axis=1 means: apply the function to each ROW (not each column)
# result_type='expand': expand the returned dict into separate columns automatically
aqi_results = df_pollution.apply(compute_row_aqi, axis=1, result_type='expand')

# ── Attach the AQI results back to the main DataFrame ────────────────────────
df_pollution['AQI']                    = aqi_results['AQI']
df_pollution['AQI_dominant_pollutant'] = aqi_results['AQI_dominant_pollutant']
df_pollution['AQI_n_pollutants_used']  = aqi_results['AQI_n_pollutants_used']

# ── Add category labels ───────────────────────────────────────────────────────
df_pollution['AQI_category']     = df_pollution['AQI'].apply(aqi_to_category)
df_pollution['AQI_project_tier'] = df_pollution['AQI'].apply(aqi_to_project_tier)

print("AQI computation complete")
print()

# ── Report results ────────────────────────────────────────────────────────────
total_rows = len(df_pollution)
has_aqi    = df_pollution['AQI'].notna().sum()
no_aqi     = total_rows - has_aqi

print(f"Total city-month rows : {total_rows:,}")
print(f"AQI computed          : {has_aqi:,}  ({has_aqi/total_rows*100:.1f}%)")
print(f"AQI not computed      : {no_aqi:,}   ({no_aqi/total_rows*100:.1f}%)")
print(f"(Rows without AQI: either < 3 pollutants or missing PM2.5 AND PM10 for that month)")
print()

# ── AQI category distribution ─────────────────────────────────────────────────
print("AQI category distribution (city-month level):")
cat_order = ['Good', 'Satisfactory', 'Moderate', 'Poor', 'Very Poor', 'Severe', 'Unknown']
cat_counts = df_pollution['AQI_category'].value_counts()
for cat in cat_order:
    count = cat_counts.get(cat, 0)
    pct   = count / total_rows * 100
    print(f"   {cat:15s}: {count:6,}  ({pct:.1f}%)")
print()

# ── Project 3-tier distribution ───────────────────────────────────────────────
print("Project 3-tier distribution:")
tier_counts = df_pollution['AQI_project_tier'].value_counts()
for tier in ['Good', 'Bad', 'Worst', 'Unknown']:
    count = tier_counts.get(tier, 0)
    pct   = count / total_rows * 100
    print(f"   {tier:10s}: {count:6,}  ({pct:.1f}%)")
print()

# ── Dominant pollutant summary ────────────────────────────────────────────────
print("Most common dominant pollutant (which pollutant drives the AQI most often):")
dom_counts = df_pollution['AQI_dominant_pollutant'].value_counts().head(7)
for pollutant, count in dom_counts.items():
    pct = count / has_aqi * 100
    print(f"   {pollutant:25s}: {count:6,}  ({pct:.1f}%)")

Computing AQI for every city-month row...
AQI computation complete

Total city-month rows : 10,295
AQI computed          : 8,086  (78.5%)
AQI not computed      : 2,209   (21.5%)
(Rows without AQI: either < 3 pollutants or missing PM2.5 AND PM10 for that month)

AQI category distribution (city-month level):
   Good           :  1,070  (10.4%)
   Satisfactory   :  2,827  (27.5%)
   Moderate       :  2,910  (28.3%)
   Poor           :    645  (6.3%)
   Very Poor      :    634  (6.2%)
   Severe         :      0  (0.0%)
   Unknown        :  2,209  (21.5%)

Project 3-tier distribution:
   Good      :  3,897  (37.9%)
   Bad       :  3,555  (34.5%)
   Worst     :    634  (6.2%)
   Unknown   :  2,209  (21.5%)

Most common dominant pollutant (which pollutant drives the AQI most often):
   PM10 (ug/m3)             :  4,785  (59.2%)
   PM2.5 (ug/m3)            :  2,518  (31.1%)
   CO (mg/m3)               :    416  (5.1%)
   Ozone (ug/m3)            :    205  (2.5%)
   NO2 (ug/m3)              :  

---
## Cell 6 — AQI Sanity Checks

We verify the AQI values are in a physically plausible range for India.  
We also check a sample of manually verifiable rows.

In [6]:
print("=" * 55)
print("AQI SANITY CHECKS")
print("=" * 55)

aqi_vals = df_pollution['AQI'].dropna()

# ── Check 1: AQI value range ──────────────────────────────────────────────────
print("\n[CHECK 1] AQI value range")
print(f"   Min    : {aqi_vals.min():.1f}")
print(f"   Mean   : {aqi_vals.mean():.1f}")
print(f"   Median : {aqi_vals.median():.1f}")
print(f"   Max    : {aqi_vals.max():.1f}")
if aqi_vals.max() > 500:
    print("   WARNING: AQI above 500 — clip logic in compute_sub_index may have failed")
elif aqi_vals.min() < 0:
    print("   WARNING: Negative AQI — concentration data issue")
else:
    print("   OK: All AQI values in valid 0-500 range")

# ── Check 2: Delhi should have high AQI (well-documented) ────────────────────
print("\n[CHECK 2] Delhi AQI (should be high — well documented)")
delhi_aqi = df_pollution[
    (df_pollution['city'].str.contains('Delhi', case=False, na=False)) &
    (df_pollution['AQI'].notna())
]['AQI']
if len(delhi_aqi) > 0:
    print(f"   Delhi mean AQI across all months: {delhi_aqi.mean():.1f}")
    print(f"   Delhi max  AQI                  : {delhi_aqi.max():.1f}")
    if delhi_aqi.mean() > 100:
        print("   OK: Delhi mean AQI > 100 (Moderate or worse) — consistent with known data")
    else:
        print("   NOTE: Delhi mean AQI < 100 — this is monthly average data, so lower than daily peaks is expected")
else:
    print("   Delhi not found in data")

# ── Check 3: Winter months should have higher AQI than summer ────────────────
# WHY: India's PM2.5 peaks in winter (Oct-Feb) due to crop burning + cold air trapping
print("\n[CHECK 3] Seasonal pattern (winter should have higher AQI than summer)")
seasonal = (
    df_pollution[df_pollution['AQI'].notna()]
    .groupby('month')['AQI']
    .mean()
    .round(1)
)
for month, aqi in seasonal.items():
    season = '(winter)' if month in [11, 12, 1, 2] else '(summer)' if month in [4, 5, 6] else '       '
    print(f"   Month {month:2d} {season}: mean AQI = {aqi}")

winter_mean = seasonal[[11, 12, 1, 2]].mean() if all(m in seasonal.index for m in [11, 12, 1, 2]) else None
summer_mean = seasonal[[4, 5, 6]].mean()      if all(m in seasonal.index for m in [4, 5, 6])      else None
if winter_mean and summer_mean:
    if winter_mean > summer_mean:
        print(f"\n   OK: Winter mean ({winter_mean:.1f}) > Summer mean ({summer_mean:.1f}) — seasonal pattern confirmed")
    else:
        print(f"\n   NOTE: Winter mean ({winter_mean:.1f}) not higher than Summer ({summer_mean:.1f}) — check if monsoon months are affecting this")

print()
print("=" * 55)

AQI SANITY CHECKS

[CHECK 1] AQI value range
   Min    : 10.0
   Mean   : 124.9
   Median : 102.3
   Max    : 400.0
   OK: All AQI values in valid 0-500 range

[CHECK 2] Delhi AQI (should be high — well documented)
   Delhi mean AQI across all months: 227.0
   Delhi max  AQI                  : 398.7
   OK: Delhi mean AQI > 100 (Moderate or worse) — consistent with known data

[CHECK 3] Seasonal pattern (winter should have higher AQI than summer)
   Month  1 (winter): mean AQI = 172.8
   Month  2 (winter): mean AQI = 147.6
   Month  3        : mean AQI = 123.3
   Month  4 (summer): mean AQI = 120.0
   Month  5 (summer): mean AQI = 109.6
   Month  6 (summer): mean AQI = 95.5
   Month  7        : mean AQI = 66.2
   Month  8        : mean AQI = 63.5
   Month  9        : mean AQI = 67.7
   Month 10        : mean AQI = 125.8
   Month 11 (winter): mean AQI = 185.6
   Month 12 (winter): mean AQI = 184.8

   OK: Winter mean (172.7) > Summer mean (108.4) — seasonal pattern confirmed



---
## Cell 7 — Save air_pollution_with_aqi.csv

We save the full city × month dataset with the new AQI columns added.

In [7]:
aqi_output_path = OUTPUT_DIR / 'air_pollution_with_aqi.csv'
df_pollution.to_csv(aqi_output_path, index=False)

import os
size_mb = os.path.getsize(aqi_output_path) / 1e6

print(f"air_pollution_with_aqi.csv saved")
print(f"   Path  : {aqi_output_path}")
print(f"   Shape : {df_pollution.shape[0]:,} rows x {df_pollution.shape[1]} columns")
print(f"   Size  : {size_mb:.2f} MB")
print()
print(f"New columns added vs air_pollution_master.csv:")
print(f"   AQI                    — numeric AQI value (0-500, NaN if insufficient data)")
print(f"   AQI_dominant_pollutant — which pollutant drove the AQI for this month")
print(f"   AQI_n_pollutants_used  — how many pollutants contributed to this AQI")
print(f"   AQI_category           — CPCB 6-category label")
print(f"   AQI_project_tier       — project 3-tier label (Good/Bad/Worst)")

air_pollution_with_aqi.csv saved
   Path  : ..\outputs\air_pollution_with_aqi.csv
   Shape : 10,295 rows x 102 columns
   Size  : 7.34 MB

New columns added vs air_pollution_master.csv:
   AQI                    — numeric AQI value (0-500, NaN if insufficient data)
   AQI_dominant_pollutant — which pollutant drove the AQI for this month
   AQI_n_pollutants_used  — how many pollutants contributed to this AQI
   AQI_category           — CPCB 6-category label
   AQI_project_tier       — project 3-tier label (Good/Bad/Worst)


---
## Cell 8 — Aggregate AQI to State × Year Level

The disease data (and `combined_master.csv`) is at state × year level.  
To update `combined_master.csv` we need to collapse AQI from city-month to state-year.

**What we compute per (state, year):**
- `AQI_annual_mean` — mean AQI across all city-months with a valid AQI
- `pct_months_Good` — % of city-months in the Good tier (AQI 0–100)
- `pct_months_Bad` — % of city-months in the Bad tier (AQI 101–300)
- `pct_months_Worst` — % of city-months in the Worst tier (AQI 301+)
- `n_months_with_aqi` — how many city-months had a valid AQI (data density indicator)

In [8]:
print("Aggregating AQI to state-year level...")

# ── Work only with rows that have a valid AQI ─────────────────────────────────
# WHY: Rows with AQI=NaN (insufficient pollutant data) must be excluded from
# the percentage calculations — otherwise they would inflate the denominator
# and make the percentages artificially low.
df_has_aqi = df_pollution[df_pollution['AQI'].notna()].copy()

# ── Step 1: Mean AQI per state-year ──────────────────────────────────────────
aqi_mean = (
    df_has_aqi
    .groupby(['state', 'year'])['AQI']
    .agg(
        AQI_annual_mean    = 'mean',
        n_months_with_aqi  = 'count',
    )
    .reset_index()
)
aqi_mean['AQI_annual_mean'] = aqi_mean['AQI_annual_mean'].round(1)

# ── Step 2: Percentage of months in each project tier ────────────────────────
# For each (state, year, tier), count how many city-months fall in that tier
tier_counts = (
    df_has_aqi
    .groupby(['state', 'year', 'AQI_project_tier'])
    .size()
    .reset_index(name='count')
)

# Total city-months with AQI per state-year (denominator for percentages)
tier_totals = (
    df_has_aqi
    .groupby(['state', 'year'])
    .size()
    .reset_index(name='total')
)

tier_counts = tier_counts.merge(tier_totals, on=['state', 'year'], how='left')
tier_counts['pct'] = (tier_counts['count'] / tier_counts['total'] * 100).round(1)

# Pivot so each tier becomes a column
# WHY: We want one row per (state, year) with columns pct_months_Good etc.
tier_pivot = tier_counts.pivot_table(
    index   = ['state', 'year'],
    columns = 'AQI_project_tier',
    values  = 'pct',
    fill_value = 0.0    # States with zero months in a tier get 0%, not NaN
).reset_index()

# Rename columns clearly
tier_pivot.columns.name = None  # Remove the 'AQI_project_tier' column group name
rename_map = {}
for col in tier_pivot.columns:
    if col in ['Good', 'Bad', 'Worst', 'Unknown']:
        rename_map[col] = f'pct_months_{col}'
tier_pivot.rename(columns=rename_map, inplace=True)

# ── Step 3: Merge mean AQI + tier percentages ─────────────────────────────────
aqi_state_year = aqi_mean.merge(tier_pivot, on=['state', 'year'], how='left')

# Ensure all three tier columns exist even if some states have no months in that tier
for col in ['pct_months_Good', 'pct_months_Bad', 'pct_months_Worst']:
    if col not in aqi_state_year.columns:
        aqi_state_year[col] = 0.0

# Add the dominant AQI category per state-year (whichever tier has the highest %)
def dominant_tier(row):
    tiers = {
        'Good'  : row.get('pct_months_Good',  0),
        'Bad'   : row.get('pct_months_Bad',   0),
        'Worst' : row.get('pct_months_Worst', 0),
    }
    return max(tiers, key=tiers.get)

aqi_state_year['AQI_dominant_tier'] = aqi_state_year.apply(dominant_tier, axis=1)

print(f"State-year AQI aggregation complete")
print(f"   Shape: {aqi_state_year.shape[0]:,} rows x {aqi_state_year.shape[1]} columns")
print()
print("Columns produced:")
for col in aqi_state_year.columns:
    print(f"   {col}")
print()
print("Sample (first 4 rows):")
print(aqi_state_year.head(4).to_string())

Aggregating AQI to state-year level...
State-year AQI aggregation complete
   Shape: 176 rows x 8 columns

Columns produced:
   state
   year
   AQI_annual_mean
   n_months_with_aqi
   pct_months_Bad
   pct_months_Good
   pct_months_Worst
   AQI_dominant_tier

Sample (first 4 rows):
            state  year  AQI_annual_mean  n_months_with_aqi  pct_months_Bad  pct_months_Good  pct_months_Worst AQI_dominant_tier
0  Andhra Pradesh  2016             86.7                  6            50.0             50.0               0.0              Good
1  Andhra Pradesh  2017             98.9                 28            39.3             60.7               0.0              Good
2  Andhra Pradesh  2018             86.1                 60            31.7             68.3               0.0              Good
3  Andhra Pradesh  2019             80.7                 55            23.6             76.4               0.0              Good


---
## Cell 9 — Update combined_master.csv with AQI State-Year Columns

The `combined_master.csv` from Phase 3 has disease data + raw pollutant means.  
We now add the AQI state-year columns to it, giving analysts a single file  
that has everything: disease deaths, pollutant concentrations, and AQI category.

**Join key:** `location_name` (combined) = `state` (aqi_state_year) + `year`.

In [9]:
# ── Rename 'state' to 'location_name' in the AQI aggregation for joining ──────
# WHY: combined_master uses 'location_name' as the state column (Phase 3 naming).
# We match that naming convention here so the merge key is consistent.
aqi_for_merge = aqi_state_year.rename(columns={'state': 'location_name'})

rows_before = len(df_combined)

# ── Merge AQI columns into combined_master ────────────────────────────────────
# how='left': keep ALL combined_master rows. Add AQI where available.
# States without AQI data (e.g. Goa) will get NaN AQI columns — correct.
df_combined_updated = df_combined.merge(
    aqi_for_merge,
    on=['location_name', 'year'],
    how='left'
)

rows_after = len(df_combined_updated)

print(f"Rows before merge: {rows_before:,}")
print(f"Rows after merge : {rows_after:,}")

if rows_after != rows_before:
    print(f"WARNING: Row count changed by {abs(rows_after - rows_before):,}")
    print(f"This means aqi_state_year has duplicate (state, year) rows — check Cell 8")
else:
    print("OK: Row count unchanged — merge produced no duplicates")

print()
print(f"Columns in updated combined_master: {df_combined_updated.shape[1]}")
print(f"(Was: {df_combined.shape[1]} — added {df_combined_updated.shape[1] - df_combined.shape[1]} AQI columns)")

# ── Check AQI coverage in the updated combined file ───────────────────────────
has_aqi_combined = df_combined_updated['AQI_annual_mean'].notna().sum()
pct_coverage     = has_aqi_combined / len(df_combined_updated) * 100
print(f"\nRows with AQI_annual_mean in combined file: {has_aqi_combined:,} ({pct_coverage:.1f}%)")

Rows before merge: 18,228
Rows after merge : 18,228
OK: Row count unchanged — merge produced no duplicates

Columns in updated combined_master: 49
(Was: 43 — added 6 AQI columns)

Rows with AQI_annual_mean in combined file: 7,392 (40.6%)


---
## Cell 10 — Save Updated combined_master.csv

In [10]:
# ── Save the updated combined_master.csv ─────────────────────────────────────
# WHY: We overwrite the Phase 3 version. The Phase 3 version is the base;
# this Phase 4 version is strictly better (all Phase 3 data + AQI columns).
# The Phase 4 combined_master.csv is the one all future analysis should use.
df_combined_updated.to_csv(COMBINED_MASTER_PATH, index=False)

import os
size_mb = os.path.getsize(COMBINED_MASTER_PATH) / 1e6

print(f"combined_master.csv updated and saved")
print(f"   Path  : {COMBINED_MASTER_PATH}")
print(f"   Shape : {df_combined_updated.shape[0]:,} rows x {df_combined_updated.shape[1]} columns")
print(f"   Size  : {size_mb:.2f} MB")
print()
print("New columns added to combined_master.csv (vs Phase 3 version):")
new_cols = [c for c in df_combined_updated.columns if c not in df_combined.columns]
for col in new_cols:
    print(f"   {col}")

combined_master.csv updated and saved
   Path  : ..\outputs\combined_master.csv
   Shape : 18,228 rows x 49 columns
   Size  : 6.36 MB

New columns added to combined_master.csv (vs Phase 3 version):
   AQI_annual_mean
   n_months_with_aqi
   pct_months_Bad
   pct_months_Good
   pct_months_Worst
   AQI_dominant_tier


---
## Cell 11 — Phase 4 Summary and Next Steps

In [11]:
print("=" * 60)
print("PHASE 4 COMPLETE — SUMMARY")
print("=" * 60)
print()

print("Files saved:")
print(f"  outputs/air_pollution_with_aqi.csv")
print(f"      City × month data with AQI columns added")
print(f"      {df_pollution.shape[0]:,} rows x {df_pollution.shape[1]} columns")
print()
print(f"  outputs/combined_master.csv  (UPDATED from Phase 3)")
print(f"      Disease + pollution + AQI — the complete analysis table")
print(f"      {df_combined_updated.shape[0]:,} rows x {df_combined_updated.shape[1]} columns")
print()

print("AQI columns in air_pollution_with_aqi.csv:")
print("  AQI                    — numeric (0-500, NaN if < 3 pollutants or no PM2.5/PM10)")
print("  AQI_dominant_pollutant — pollutant with highest sub-index")
print("  AQI_n_pollutants_used  — number of pollutants in the AQI calculation")
print("  AQI_category           — Good / Satisfactory / Moderate / Poor / Very Poor / Severe")
print("  AQI_project_tier       — Good (0-100) / Bad (101-300) / Worst (301+)")
print()

print("AQI columns added to combined_master.csv:")
print("  AQI_annual_mean        — mean AQI across all city-months for this state-year")
print("  n_months_with_aqi      — how many city-months contributed to the state-year average")
print("  pct_months_Good        — % of city-months in Good tier (AQI 0-100)")
print("  pct_months_Bad         — % of city-months in Bad tier (AQI 101-300)")
print("  pct_months_Worst       — % of city-months in Worst tier (AQI 301+)")
print("  AQI_dominant_tier      — which tier had the most months for this state-year")
print()

print("LIMITATION to state in the project report:")
print("  CPCB AQI is designed for 24-hr averages (PM2.5, PM10, NO2, SO2, NH3)")
print("  and 8-hr averages (CO, Ozone). Our data uses monthly averages.")
print("  Monthly AQI will underestimate actual daily peak AQI.")
print("  Results are valid for RELATIVE comparison between states and years,")
print("  but should NOT be equated to official CPCB daily AQI figures.")
print()

print("-" * 60)
print("NEXT: Phase 5 — ML Prediction Models (phase_5_ml.ipynb)")
print()
print("  Two separate ML tasks:")
print("  Task A — Predict next month's AQI / PM2.5 for a city")
print("    Input : air_pollution_with_aqi.csv (city x month)")
print("    Features: historical PM2.5, AT, RH, WS, RF, BP, month, year")
print("    Target  : next month's AQI or PM2.5_mean")
print("    Eligible: tier_1 and tier_2 cities only")
print()
print("  Task B — Predict next year's deaths from disease")
print("    Input : combined_master.csv (state x year)")
print("    Features: AQI_annual_mean, PM2.5 mean, pct_months_Worst")
print("    Target  : val (deaths) for each disease cause")
print("    Filter  : join_quality == 'full', metric_id == 1 (Number)")
print("=" * 60)

PHASE 4 COMPLETE — SUMMARY

Files saved:
  outputs/air_pollution_with_aqi.csv
      City × month data with AQI columns added
      10,295 rows x 102 columns

  outputs/combined_master.csv  (UPDATED from Phase 3)
      Disease + pollution + AQI — the complete analysis table
      18,228 rows x 49 columns

AQI columns in air_pollution_with_aqi.csv:
  AQI                    — numeric (0-500, NaN if < 3 pollutants or no PM2.5/PM10)
  AQI_dominant_pollutant — pollutant with highest sub-index
  AQI_n_pollutants_used  — number of pollutants in the AQI calculation
  AQI_category           — Good / Satisfactory / Moderate / Poor / Very Poor / Severe
  AQI_project_tier       — Good (0-100) / Bad (101-300) / Worst (301+)

AQI columns added to combined_master.csv:
  AQI_annual_mean        — mean AQI across all city-months for this state-year
  n_months_with_aqi      — how many city-months contributed to the state-year average
  pct_months_Good        — % of city-months in Good tier (AQI 0-100)
  p